# Copyright 2022 Cognite AS
# Part 1 - Calculating the thermal resistance

For Part 1 of the Hands-On Exercise, we will retrieve heat exchanger data from CDF using the Python SDK and use it to calculate the thermal resistance. The steps include the following:

1. Authentication and Client Creation
2. Finding data related to the system
3. Retrieve the data
4. Visualise the data
5. Calculation of the Thermal Resistance


## 1. Authentication and Client Creation

Here we'll create the Cognite client using an interactive flow. Once this client is created, this will be the main way to communicate with the Cognite Data Fusion.

Import and create a client to communicate with the Cognite Data Fusion (CDF)

In [ ]:
from cognite.client import CogniteClient, ClientConfig
from cognite.client.credentials import Token
from msal import PublicClientApplication

# Contact Project Administrator to get these
TENANT_ID = "48d5043c-cf70-4c49-881c-c638f5796997"
CLIENT_ID = "1c224c70-1df2-4f05-964a-753a6731748a"
CDF_CLUSTER = "westeurope-1"
COGNITE_PROJECT = "ds-cognitefunctions"

SCOPES = [f"https://{CDF_CLUSTER}.cognitedata.com/.default"]
TOKEN_URL = f"https://login.microsoftonline.com/{TENANT_ID}/oauth2/v2.0/token"
AUTHORITY_HOST_URI = "https://login.microsoftonline.com"
AUTHORITY_URI = AUTHORITY_HOST_URI + "/" + TENANT_ID
PORT = 53000


def authenticate_azure():

    app = PublicClientApplication(client_id=CLIENT_ID, authority=AUTHORITY_URI)

    # interactive login - make sure you have http://localhost:port in Redirect URI in App Registration as type "Mobile and desktop applications"
    creds = app.acquire_token_interactive(scopes=SCOPES, port=PORT)
    return creds


creds = authenticate_azure()

cnf = ClientConfig(
    client_name="my-special-client",
    project=COGNITE_PROJECT,
    credentials=Token(creds["access_token"]),
    base_url=f"https://{CDF_CLUSTER}.cognitedata.com",
)
client = CogniteClient(cnf)

This is just one of the ways to create the client, there are many other ways e.g. Device code etc
Guide for authentication and creating the client is available here https://github.com/cognitedata/python-oidc-authentication

## 2. Finding data related to the System

This can be achieved with the Hierarchy based structure in the CDF. High level System are stored as the Assets e.g. here **23-HA-9114** and **23-HA-9115** are stored as the assets. All the tags related to them are linked to the respective assets in order to access them easily.

Let's quickly get the IDs for assets we're interested in

In [ ]:
asset_name = "23-HA-9114"
assets = client.assets.search(name=asset_name)
assets[:5]

Now we can use the first two IDs to get the time series or any other data related to these two Heat Exchangers/Discharge coolers **23-HA-9114** and **23-HA-9115**

In [ ]:
client.time_series.list(asset_subtree_ids=[7690256520625010, 7860450365103602])[:5]

You can also retrieve the subtree for these Assets

In [ ]:
subtree = client.assets.retrieve_subtree(id=7690256520625010)
subtree[:5]

There are many filtering and searching options in python sdk. refer to the documentation for more details.

## 3. Retrieve the Data

Let's Fetch the Temperatures and Flow rates for the discharge coolers.
We can look up external IDs for the sensors we're interested in and use them to filter & retrieve the data we need from CDF.

Specify the start and end date , there are also options for aggregation and granularity of data.
Note : Instead of dates, we can also specify strings for start and end dates, e.g. **start="2w-ago", end="now"**



In [ ]:
column_names = [
    "T_cold_IN",
    "T_cold_OUT",
    "T_hot_IN",
    "T_hot_OUT",
    "Flow_cold",
    "Flow_hot",
]
ts_exids = [
    "pi:163657",
    "pi:163658",
    "pi:160887",
    "pi:191092",
    "pi:163374",
    "pi:160184",
]

import datetime
from datetime import timedelta

start_date = datetime.datetime(2018, 8, 1)
end_date = start_date + timedelta(days=30)

df = client.datapoints.retrieve_dataframe(
    external_id=ts_exids,
    aggregates=["average"],
    granularity="1h",
    start=start_date,
    end=end_date,
    include_aggregate_name=False,
)
df.fillna(method="ffill", inplace=True)
df.columns = column_names
df.head()

In [ ]:
client.datapoints.retrieve_dataframe(
    external_id=ts_exids,
    aggregates=["average"],
    granularity="1h",
    start=start_date,
    end=end_date,
    include_aggregate_name=False,
)

## 4. Visualize the data

Let's quickly plot the data

In [ ]:
import matplotlib.pyplot as plt

cols = df.columns

fig, axes = plt.subplots(nrows=3, ncols=2, figsize=(20, 15))
for i, col in enumerate(cols):
    df.loc[:, [col]].plot(ax=axes.ravel()[i])

## 5. Heat Exchanger Calculations
First of all let's bring the mass flow rates into same unit of measurement. (kg/seconds)

In [ ]:
df["Flow_cold"] = df["Flow_cold"] * 1000 / 3600.0
df["Flow_hot"] = df["Flow_hot"] / 3600.0
df.head()

Let's create the function for **Thermal Resistance Calculation** `TR = A * F * LMTD/Q`

where A=Area, F= Correction Factor, LMTD - Log Mean Temperature Difference,
Q - Heat Transfer Rate

The Area and Correction factor are constants that are taken from the unit specification. The **Log Mean Temperature Difference** is a dynamic value that is calculated with the inlet and outlet temperatures as follows:

`LMTD = ((T_hot_IN-T_cold_OUT)-(T_hot_OUT-T_cold_IN))/(log((T_hot_IN-T_cold_OUT)/(T_hot_OUT-T_cold_IN)))`

where T_hot_IN is the inlet temperature of the hot stream, T_hot_OUT is the outlet temperature of the hot stream, T_cold_IN is the inlet temperature of the cold stream and T_cold_OUT is the outlet temperature of the cold stream.

The **Heat Transfer Q** can be calculated using the flowrate of the hot stream and the temperature difference between the inlet and the outlet.

`Q = Flow_hot * Cp_hot *(T_hot_IN-T_hot_OUT)`

where Flow_hot is the flow rate of the hot stream and Cp_hot is its heat capacity (which in this case we are treating as a thermodynamic constant. 

Let's implement these formulas in a function!

In [ ]:
from math import log


def thermal_resistance(x):
    """Function to calculate Thermal Resistance"""
    # Some constants like Correction factor, Area and Cp values
    F = 0.8
    A = 1.0
    Cp_hot = 2.4
    # Calculate the cross temperature differences
    x["dT1"] = abs(x["T_hot_IN"] - x["T_cold_OUT"])
    x["dT2"] = abs(x["T_hot_OUT"] - x["T_cold_IN"])
    # Calculate the numerator and denominator for the thermal resistance calculation
    temp1 = A * F * (x["dT1"] - x["dT2"]) / log(x["dT1"] / x["dT2"])
    temp2 = x["Flow_hot"] * Cp_hot * (x["T_hot_IN"] - x["T_hot_OUT"])
    TR = temp1 / temp2
    return TR


df["TR"] = df.apply(lambda x: thermal_resistance(x), axis=1)
df.head()

Now this calculation result is only available in this notebook, and each time  we need the value of Thermal Resitance, we'll have to fecth the data and run the above code manually. In the case, we want to presist the result of this calculation, we need to save this in CDF.

Cognite functions can be a solution for this type of calculation and saving results in CDF to be accessible later use or in other applications. 

In [ ]:
df["TR"].plot(figsize=(15, 5))
plt.title("Thermal Resistance")